In [6]:
import pandas as pd
import numpy as np
import warnings
from scipy import optimize, sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

warnings.filterwarnings("ignore")

In [7]:
# 1. Load files
train_path = "./train.csv"
test_path = "./test.csv"
sample_path = "./sample.csv"

full_train = pd.read_csv(train_path)
test_template = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print("full train shape:", full_train.shape)
print("test shape:", test_template.shape)
print("sample shape:", sample_sub.shape)

# use labeled rows for training and the hidden rows for prediction
train_df = full_train[full_train["Score"].notna()].copy()
predict_df = full_train[full_train["Score"].isna()].copy()

# updated merge so prediction ids follow the official test file order
predict_df = test_template[["Id"]].merge(predict_df, on="Id", how="left")

train_df["Score"] = train_df["Score"].astype(int)
y = train_df["Score"].values

print("usable training rows:", train_df.shape)
print("rows to predict:", predict_df.shape)

def combine_text(df):
    # still using summary + text together, this worked better than using only one field
    return (df["Summary"].fillna("") + " " + df["Text"].fillna("")).str.lower()

X_train_raw = combine_text(train_df)
X_pred_raw = combine_text(predict_df)

full train shape: (139753, 9)
test shape: (13976, 2)
sample shape: (13976, 2)
usable training rows: (125777, 9)
rows to predict: (13976, 9)


In [8]:
# 2. Text features
# this part was expanded from a more basic tf-idf version
print("Vectorizing text features...")

# kept a word-level tf-idf because it is still the main signal
# compared with a simpler setup before, adding 3-grams helped a bit
word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=100000,
    min_df=5,
    stop_words="english",
    sublinear_tf=True
)

# later I also added a char-level tf-idf for misspellings / short patterns
# keeping it because it gave a small boost without changing the overall pipeline too much
char_vec = TfidfVectorizer(
    ngram_range=(3, 6),
    max_features=50000,
    min_df=10,
    analyzer="char",
    sublinear_tf=True
)

X_train_word = word_vec.fit_transform(X_train_raw)
X_pred_word = word_vec.transform(X_pred_raw)

X_train_char = char_vec.fit_transform(X_train_raw)
X_pred_char = char_vec.transform(X_pred_raw)

# updated from single matrix to combined word + char features
X_train = sparse.hstack([X_train_word, X_train_char]).tocsr()
X_pred = sparse.hstack([X_pred_word, X_pred_char]).tocsr()

print("word features:", X_train_word.shape[1])
print("char features:", X_train_char.shape[1])
print("total features:", X_train.shape[1])

Vectorizing text features...
word features: 100000
char features: 50000
total features: 150000


In [10]:
# 3. Cross-validation and model blending
# keeping the 5-fold structure, then gradually adding a couple of extra models
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = np.zeros((len(train_df), 5))
test_probs = np.zeros((len(predict_df), 5))

print("Starting 5-fold training...")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), 1):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # first model I kept from the earlier version because it was pretty fast
    m1 = SGDClassifier(
        loss="log_loss",
        alpha=2e-5,
        max_iter=3000,
        tol=1e-4,
        random_state=42
    )
    m1.fit(X_tr, y_tr)

    # then tuned logistic regression a bit instead of replacing it completely
    m2 = LogisticRegression(
        C=0.8,
        solver="lbfgs",
        max_iter=2000
    )
    m2.fit(X_tr, y_tr)

    # added ComplementNB as a light third model
    m3 = ComplementNB(alpha=0.1)
    m3.fit(X_tr, y_tr)

    # updated blend weights after some trial and error
    va_prob = (
        0.6 * m1.predict_proba(X_va) +
        0.2 * m2.predict_proba(X_va) +
        0.2 * m3.predict_proba(X_va)
    )
    oof_probs[va_idx] = va_prob

    test_prob = (
        0.6 * m1.predict_proba(X_pred) +
        0.2 * m2.predict_proba(X_pred) +
        0.2 * m3.predict_proba(X_pred)
    )
    test_probs += test_prob / skf.n_splits

    print(f"fold {fold} complete")

Starting 5-fold training...
fold 1 complete
fold 2 complete
fold 3 complete
fold 4 complete
fold 5 complete


In [11]:
# 4. QWK optimization
# keeping this final step because plain argmax was a bit worse in my earlier runs
oof_cont = np.dot(oof_probs, np.array([1, 2, 3, 4, 5]))
test_cont = np.dot(test_probs, np.array([1, 2, 3, 4, 5]))

class OptimizedRounder:
    def __init__(self):
        self.coeff = [1.5, 2.5, 3.5, 4.5]

    def _kappa_loss(self, coeff, X, y):
        X_p = pd.cut(
            X,
            [-np.inf] + list(np.sort(coeff)) + [np.inf],
            labels=[1, 2, 3, 4, 5]
        )
        return -cohen_kappa_score(y, X_p, weights="quadratic")

    def fit(self, X, y):
        # kept the optimization simple here
        res = optimize.minimize(
            self._kappa_loss,
            self.coeff,
            args=(X, y),
            method="Nelder-Mead"
        )
        self.coeff = res.x

    def predict(self, X):
        return pd.cut(
            X,
            [-np.inf] + list(np.sort(self.coeff)) + [np.inf],
            labels=[1, 2, 3, 4, 5]
        )

print("Running QWK optimization...")
optR = OptimizedRounder()
optR.fit(oof_cont, y)
final_predictions = optR.predict(test_cont)

print("optimized thresholds:", np.sort(optR.coeff))

Running QWK optimization...
optimized thresholds: [2.52518298 3.08259762 3.54546434 4.12464074]


In [12]:
# 5. Build submission
submission = sample_sub.copy()
submission["Score"] = final_predictions.astype(int)
submission.to_csv("submission.csv", index=False)

print("\nSaved submission.csv")
print("Prediction distribution:")
print(submission["Score"].value_counts().sort_index())


Saved submission.csv
Prediction distribution:
Score
1     742
2     941
3    1540
4    3362
5    7391
Name: count, dtype: int64
